# Drawdown Diagnostics — Module 07

This notebook diagnoses the large out-of-sample drawdown without rerunning the strategy. It isolates the exact peak-to-trough episode, identifies which trades/pairs drove the loss, measures concentration, and checks whether residual stationarity weakened around the event. The ADF section is diagnostic only.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)

## 1. Load completed outputs

In [ ]:
trades = pd.read_parquet("data/processed/option_trades.parquet")
equity_curve = pd.read_parquet("data/processed/equity_curve.parquet")
skipped_signals = pd.read_parquet("data/processed/skipped_signals.parquet")
train_prices = pd.read_parquet("data/processed/train_prices.parquet")
test_prices = pd.read_parquet("data/processed/test_prices.parquet")
cointegrated_pairs = pd.read_parquet("data/processed/cointegrated_pairs.parquet")

trades["entry_date"] = pd.to_datetime(trades["entry_date"])
trades["exit_date"] = pd.to_datetime(trades["exit_date"])
equity_curve.index = pd.to_datetime(equity_curve.index)
train_prices.index = pd.to_datetime(train_prices.index)
test_prices.index = pd.to_datetime(test_prices.index)
equity_curve = equity_curve.sort_index()

print(f"Trades: {len(trades)}")
print(f"OOS: {equity_curve.index.min().date()} -> {equity_curve.index.max().date()}")
print(f"Initial equity: ${equity_curve['equity'].iloc[0]:,.2f}")
print(f"Final equity:   ${equity_curve['equity'].iloc[-1]:,.2f}")

## 2. Exact maximum drawdown

In [ ]:
eq = equity_curve.copy()
eq["running_peak"] = eq["equity"].cummax()
eq["drawdown"] = eq["equity"] / eq["running_peak"] - 1.0
eq["daily_equity_change"] = eq["equity"].diff()

trough_date = eq["drawdown"].idxmin()
peak_date = eq.loc[:trough_date, "equity"].idxmax()
peak_equity = float(eq.loc[peak_date, "equity"])
trough_equity = float(eq.loc[trough_date, "equity"])
dollar_drawdown = trough_equity - peak_equity
pct_drawdown = trough_equity / peak_equity - 1
drawdown_eq = eq.loc[peak_date:trough_date].copy()

print(f"Peak date:     {peak_date.date()}")
print(f"Peak equity:   ${peak_equity:,.2f}")
print(f"Trough date:   {trough_date.date()}")
print(f"Trough equity: ${trough_equity:,.2f}")
print(f"Dollar loss:   ${dollar_drawdown:,.2f}")
print(f"Drawdown:      {pct_drawdown:.2%}")

plt.figure(figsize=(13,5))
plt.plot(eq.index, eq["equity"])
plt.axvline(peak_date, linestyle="--")
plt.axvline(trough_date, linestyle="--")
plt.title("OOS Equity Curve and Maximum Drawdown")
plt.xlabel("Date")
plt.ylabel("Equity ($)")
plt.grid(alpha=0.25)
plt.show()

## 3. Worst daily equity changes

In [ ]:
worst_days = drawdown_eq[["equity","cash","n_open_positions","daily_equity_change"]].dropna().sort_values("daily_equity_change")
display(worst_days.head(30))
print(f"Sum of 20 worst daily changes: ${worst_days.head(20)['daily_equity_change'].sum():,.2f}")

## 4. Trades active during the peak-to-trough episode

In [ ]:
dd_trades = trades[(trades["entry_date"] <= trough_date) & (trades["exit_date"] >= peak_date)].copy()
dd_trades["entered_during_dd"] = dd_trades["entry_date"].between(peak_date, trough_date)
dd_trades["exited_during_dd"] = dd_trades["exit_date"].between(peak_date, trough_date)

print(f"Overlapping trades: {len(dd_trades)}")
print(f"Entered during DD: {dd_trades['entered_during_dd'].sum()}")
print(f"Exited during DD:  {dd_trades['exited_during_dd'].sum()}")
print(f"Realized PnL:      ${dd_trades['pnl'].sum():,.2f}")
print(f"Win rate:          {(dd_trades['pnl'] > 0).mean():.2%}")

display(dd_trades.sort_values("pnl")[["pair","entry_date","exit_date","entry_z","entry_premium","exit_value","pnl","trade_return","exit_reason","entered_during_dd","exited_during_dd"]].head(40))

## 5. Loss clustering by exit date

In [ ]:
realized_in_dd = trades[trades["exit_date"].between(peak_date, trough_date)].copy()
exit_damage = realized_in_dd.groupby("exit_date").agg(n_exits=("pnl","size"), n_losers=("pnl", lambda x: (x < 0).sum()), realized_pnl=("pnl","sum"), gross_losses=("pnl", lambda x: x[x < 0].sum()), gross_winners=("pnl", lambda x: x[x > 0].sum())).sort_values("realized_pnl")
display(exit_damage.head(30))

## 6. Largest individual losers

In [ ]:
largest_losers = trades.loc[trades["pnl"] < 0].copy().sort_values("pnl")
display(largest_losers[["pair","entry_date","exit_date","entry_z","convergence_horizon_trading_days","option_calendar_dte","dependent_contracts","independent_contracts","entry_premium","exit_value","pnl","trade_return","exit_reason"]].head(40))
print(f"10 largest losses: ${largest_losers.head(10)['pnl'].sum():,.2f}")
print(f"20 largest losses: ${largest_losers.head(20)['pnl'].sum():,.2f}")
print(f"Peak-to-trough equity loss: ${dollar_drawdown:,.2f}")

## 7. Pair-level performance and win rates

In [ ]:
pair_stats = trades.groupby("pair").agg(n_trades=("pnl","size"), total_pnl=("pnl","sum"), mean_pnl=("pnl","mean"), median_return=("trade_return","median"), win_rate=("pnl", lambda x: (x > 0).mean()), total_premium=("entry_premium","sum"))
print("Worst pairs by total PnL")
display(pair_stats.sort_values("total_pnl").head(20))
print("Lowest win-rate pairs, minimum 3 trades")
display(pair_stats[pair_stats["n_trades"] >= 3].sort_values(["win_rate","total_pnl"]).head(20))

## 8. Pair damage specifically during the drawdown

In [ ]:
dd_pair_stats = dd_trades.groupby("pair").agg(n_overlapping_trades=("pnl","size"), total_realized_pnl=("pnl","sum"), mean_return=("trade_return","mean"), win_rate=("pnl", lambda x: (x > 0).mean()), total_entry_premium=("entry_premium","sum")).sort_values("total_realized_pnl")
display(dd_pair_stats.head(25))

## 9. Portfolio concentration during the drawdown

In [ ]:
print(f"Mean open positions:    {drawdown_eq['n_open_positions'].mean():.2f}")
print(f"Median open positions:  {drawdown_eq['n_open_positions'].median():.0f}")
print(f"Maximum open positions: {drawdown_eq['n_open_positions'].max():.0f}")
print(f"Minimum cash:           ${drawdown_eq['cash'].min():,.2f}")

plt.figure(figsize=(13,4))
plt.plot(drawdown_eq.index, drawdown_eq["n_open_positions"])
plt.title("Open Positions During Maximum Drawdown")
plt.xlabel("Date")
plt.ylabel("Open positions")
plt.grid(alpha=0.25)
plt.show()

## 10. Diagnostic trailing ADF tests

Keeps each pair's original frozen alpha and beta and tests its residual spread over a trailing 252-trading-day window, evaluated about every 126 trading days. This is diagnostic only.

In [ ]:
all_prices = pd.concat([train_prices, test_prices]).sort_index()
all_prices = all_prices[~all_prices.index.duplicated(keep="last")]
cp = cointegrated_pairs.copy()
if "pair" not in cp.columns:
    cp["pair"] = cp["dependent"].astype(str) + "-" + cp["independent"].astype(str)
cp = cp.drop_duplicates("pair").set_index("pair")

WINDOW = 252
STEP = 126
diagnostic_pairs = sorted(dd_trades["pair"].unique())
rows = []

for pair in diagnostic_pairs:
    if pair not in cp.index:
        continue
    r = cp.loc[pair]
    dep, indep = r["dependent"], r["independent"]
    alpha, beta = float(r["alpha"]), float(r["beta"])
    px = all_prices[[dep, indep]].dropna()
    spread = np.log(px[dep]) - alpha - beta * np.log(px[indep])
    oos_positions = np.where(spread.index >= test_prices.index.min())[0]
    for pos in oos_positions[::STEP]:
        if pos + 1 < WINDOW:
            continue
        window = spread.iloc[pos-WINDOW+1:pos+1].dropna()
        if len(window) < WINDOW:
            continue
        adf_stat, pvalue, *_ = adfuller(window, autolag="AIC")
        rows.append({"pair":pair,"date":spread.index[pos],"adf_stat":adf_stat,"adf_pvalue":pvalue,"stationary_5pct":pvalue < 0.05})

rolling_adf = pd.DataFrame(rows)
display(rolling_adf.head())

## 11. Stationarity status before the peak and trough

In [ ]:
def latest_adf_before(df, date):
    x = df[df["date"] <= pd.Timestamp(date)].copy()
    if x.empty:
        return pd.DataFrame()
    return x.sort_values("date").groupby("pair", as_index=False).tail(1).set_index("pair")

adf_pre_peak = latest_adf_before(rolling_adf, peak_date)
adf_pre_trough = latest_adf_before(rolling_adf, trough_date)
comparison = dd_pair_stats.copy()

if not adf_pre_peak.empty:
    comparison = comparison.join(adf_pre_peak[["date","adf_pvalue","stationary_5pct"]].rename(columns={"date":"adf_date_pre_peak","adf_pvalue":"adf_pvalue_pre_peak","stationary_5pct":"stationary_pre_peak"}), how="left")
if not adf_pre_trough.empty:
    comparison = comparison.join(adf_pre_trough[["date","adf_pvalue","stationary_5pct"]].rename(columns={"date":"adf_date_pre_trough","adf_pvalue":"adf_pvalue_pre_trough","stationary_5pct":"stationary_pre_trough"}), how="left")

display(comparison.sort_values("total_realized_pnl").head(30))

## Interpretation

If the worst drawdown pairs already show high trailing ADF p-values before the portfolio peak, that is evidence consistent with the original mean-reverting relationship weakening OOS. That would motivate a revised walk-forward pair-validity check at fixed rebalance dates using only information available then. Such a change should be treated as a new strategy specification rather than a retroactive correction to the original OOS test.